# Biohub inference

Same detect → graph upgrade → submission path as the frozen production notebook, using package APIs and `configs/infer.yaml`. Output CSV columns are `id,dataset,row_type,node_id,t,z,y,x,source_id,target_id`.

In [ ]:
from biohub.infer.run import assemble_submission, build_tracking, detect_job
from biohub.paths import PROJECT_ROOT
from biohub.utils.yaml_config import load_yaml, resolve_path

CFG = load_yaml(PROJECT_ROOT / 'configs' / 'infer.yaml')
config_path = PROJECT_ROOT / 'configs' / 'infer.yaml'
movie = '44b6_0113de3b'
work_dir = PROJECT_ROOT / 'runs' / 'notebook_infer' / 'workdir'
work_dir.mkdir(parents=True, exist_ok=True)
tracking = build_tracking(
    config_path=config_path,
    bundle_dir=resolve_path(CFG['bundle_dir']),
    test_dir=resolve_path(CFG['train_dir']),
    work_dir=work_dir,
)
tracking

In [ ]:
from biohub.modules.graph.upgrade import GraphUpgrade

upgrade = GraphUpgrade(tracking)
upgrade.init_unigraft_runtime()
upgrade.init_live_v2_runtime()
upgrade.init_ownership_runtime()
upgrade.init_edgegraft_runtime()
upgrade.init_candidategraft_runtime()
upgrade.load_motion_corrector()
upgrade.load_deepcenter()
upgrade.csv_columns

In [ ]:
from biohub.modules.detect import predict_from_job

predictions_dir = work_dir / 'predictions'
job = detect_job(
    movie_ids=[movie],
    test_dir=tracking.paths.test_dir,
    predictions_dir=predictions_dir,
    tracking=tracking,
)
pred_dir = predict_from_job(job)
geff_path = pred_dir / f'{movie}.geff'
geff_path

In [ ]:
stats = upgrade.process_graph(geff_path, 'combined')
stats

In [ ]:
import polars as pl

from biohub.infer.run import assemble_submission

submission = tracking.paths.submission
node_rows, edge_rows = assemble_submission(
    [movie],
    tracking.paths.shard_dir,
    submission,
    upgrade.csv_columns,
)
preview = pl.read_csv(submission)
{
    'path': str(submission),
    'columns': list(preview.columns),
    'nodes': node_rows,
    'edges': edge_rows,
    'preview': preview.head(),
}

In [ ]:
from biohub.metrics.evaluate import run_evaluate

evaluation = run_evaluate(
    config=config_path,
    pred_dir=submission,
    pred_source='csv',
    panel='smoke',
    require_complete=False,
    n_workers=1,
    run_id=None,
)
evaluation